# ML-experiment – Henri

Här testar jag en modell som ska förutsäga om en aktie får högre avkastning än medianen.

## Importera bibliotek

Jag importerar biblioteken som behövs för datahantering, förbehandling, maskininlärning och utvärdering av modellerna.

In [1]:

# Datahantering
import pandas as pd
import numpy as np

# Förbehandling av data
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer

# Maskininlärningsmodeller
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Utvärdering av modeller
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)


## Läsa in data

Jag använder 2014–2016 för att träna modellen och 2017 för att validera den.

År 2018 sparas till det slutliga testet för att se hur modellen fungerar på ny data.

In [2]:
df_2014 = pd.read_csv("../Data/2014_Financial_Data.csv")
df_2015 = pd.read_csv("../Data/2015_Financial_Data.csv")
df_2016 = pd.read_csv("../Data/2016_Financial_Data.csv")
df_2017 = pd.read_csv("../Data/2017_Financial_Data.csv")

## Kontrollera kolumner

Jag kontrollerar vilka kolumner som finns i träningsdatan.

In [3]:
print("2014:", df_2014.shape[1])
print("2015:", df_2015.shape[1])
print("2016:", df_2016.shape[1])

print("Skillnader mot 2014:")
print(set(df_2015.columns) - set(df_2014.columns))
print(set(df_2016.columns) - set(df_2014.columns))

2014: 225
2015: 225
2016: 225
Skillnader mot 2014:
{'2016 PRICE VAR [%]'}
{'2017 PRICE VAR [%]'}


## Förbereda träningsdata

Jag slår ihop åren 2014–2016 till en träningsdata. År 2017 används för validering.

In [4]:
df_2014 = df_2014.rename(columns={"2015 PRICE VAR [%]": "Next_Year_Return"})
df_2015 = df_2015.rename(columns={"2016 PRICE VAR [%]": "Next_Year_Return"})
df_2016 = df_2016.rename(columns={"2017 PRICE VAR [%]": "Next_Year_Return"})

train_df = pd.concat([df_2014, df_2015, df_2016], ignore_index=True)

print("Antal rader:", train_df.shape[0])
print("Antal kolumner:", train_df.shape[1])


Antal rader: 12725
Antal kolumner: 225


In [5]:
print([col for col in train_df.columns if "PRICE VAR" in col or "Next_Year_Return" in col])

print("Antal kolumner:", train_df.shape[1])
print("Dubbla kolumnnamn:", train_df.columns.duplicated().sum())

['Next_Year_Return']
Antal kolumner: 225
Dubbla kolumnnamn: 0


## Förbereda valideringsdata

Jag använder 2017 för att kontrollera hur modellen fungerar på data den inte tränats på.

In [6]:
df_2017 = df_2017.rename(columns={"2018 PRICE VAR [%]": "Next_Year_Return"})

print("Antal rader:", df_2017.shape[0])
print("Antal kolumner:", df_2017.shape[1])

Antal rader: 4960
Antal kolumner: 225


## Kontrollera avkastningen

Jag undersöker medianavkastningen för varje år innan jag skapar målvariabeln.

In [7]:
for year, df in [(2014, df_2014), (2015, df_2015),
                 (2016, df_2016), (2017, df_2017)]:
    print(year, "Median:", df["Next_Year_Return"].median())

2014 Median: -5.281841982079795
2015 Median: 17.27893469531646
2016 Median: 10.906861703108994
2017 Median: -13.336397171378067


## Skapa målvariabel

Jag skapar en kolumn som visar om aktiens avkastning är högre än medianen för samma år.

In [8]:
for df in [df_2014, df_2015, df_2016, df_2017]:
    median = df["Next_Year_Return"].median()
    df["Target"] = (df["Next_Year_Return"] > median).astype(int)

C:\Users\dropz\AppData\Local\Temp\ipykernel_26452\1160692380.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Target"] = (df["Next_Year_Return"] > median).astype(int)
C:\Users\dropz\AppData\Local\Temp\ipykernel_26452\1160692380.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Target"] = (df["Next_Year_Return"] > median).astype(int)
C:\Users\dropz\AppData\Local\Temp\ipykernel_26452\1160692380.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, w

In [9]:
for year, df in [(2014, df_2014), (2015, df_2015),
                 (2016, df_2016), (2017, df_2017)]:
    print(year)
    print(df["Target"].value_counts())

2014
Target
0    1904
1    1904
Name: count, dtype: int64
2015
Target
0    2060
1    2060
Name: count, dtype: int64
2016
Target
0    2399
1    2398
Name: count, dtype: int64
2017
Target
1    2480
0    2480
Name: count, dtype: int64


## Uppdatera träningsdata

Jag slår ihop träningsåren igen så att målvariabeln kommer med.

In [10]:
train_df = pd.concat([df_2014, df_2015, df_2016], ignore_index=True)

print("Antal rader:", train_df.shape[0])
print("Antal kolumner:", train_df.shape[1])
print(train_df["Target"].value_counts())

Antal rader: 12725
Antal kolumner: 226
Target
0    6363
1    6362
Name: count, dtype: int64


## Välja nyckeltal

Jag undersöker vilka finansiella nyckeltal som kan användas för att träna modellen.

In [11]:
for col in train_df.columns:
    if any(word in col.lower() for word in ["roe", "debt", "margin", "pe ratio"]):
        print(col)

Gross Margin
EBITDA Margin
EBIT Margin
Profit Margin
Free Cash Flow margin
Earnings Before Tax Margin
Net Profit Margin
Short-term debt
Long-term debt
Total debt
Net Debt
Issuance (repayment) of debt
grossProfitMargin
operatingProfitMargin
pretaxProfitMargin
netProfitMargin
debtRatio
debtEquityRatio
longtermDebtToCapitalization
totalDebtToCapitalization
cashFlowToDebtRatio
Interest Debt per Share
PE ratio
Debt to Equity
Debt to Assets
Net Debt to EBITDA
ROE
Debt Growth


## Valda nyckeltal

Jag väljer fem nyckeltal som beskriver företagens lönsamhet, värdering och skuldsättning.

In [12]:
features = [
    "ROE",
    "PE ratio",
    "Debt to Equity",
    "Net Profit Margin",
    "Gross Margin"
]

print(train_df[features].head())

      ROE  PE ratio  Debt to Equity  Net Profit Margin  Gross Margin
0  0.1664   18.7566          0.5061             0.1565        0.4754
1  0.3294   81.5526          1.5093             0.0364        0.2487
2  0.2821   12.0340          2.1007             0.0154        0.2057
3 -0.1180   28.6087         -2.7237             0.0098        0.2869
4  0.2792   18.7034          1.3445             0.1019        0.3557


## Kontrollera dataläckage

Jag kontrollerar att modellen bara använder de fem valda nyckeltalen.

In [13]:
print("Nyckeltal som modellen använder:")
print(features)

print("Antal nyckeltal:", len(features))

Nyckeltal som modellen använder:
['ROE', 'PE ratio', 'Debt to Equity', 'Net Profit Margin', 'Gross Margin']
Antal nyckeltal: 5


## Kontrollera saknade värden

Jag kontrollerar om det saknas värden i de fem nyckeltalen.

In [14]:
print(train_df[features].isna().sum())

ROE                  1243
PE ratio             1163
Debt to Equity       1228
Net Profit Margin     885
Gross Margin          619
dtype: int64


In [15]:
print(np.isinf(train_df[features]).sum())

ROE                  0
PE ratio             0
Debt to Equity       0
Net Profit Margin    0
Gross Margin         0
dtype: int64


## Hantera saknade värden

Jag använder medianen från träningsdatan för att fylla i saknade värden.

In [16]:

imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(train_df[features])
X_val = imputer.transform(df_2017[features])

In [17]:
print("Saknade värden i träning:", np.isnan(X_train).sum())
print("Saknade värden i validering:", np.isnan(X_val).sum())

Saknade värden i träning: 0
Saknade värden i validering: 0


## Förbereda målvariabler

Jag använder Target för att visa modellen vad den ska förutsäga.

In [18]:
y_train = train_df["Target"]
y_val = df_2017["Target"]

print("Träning:", y_train.shape)
print("Validering:", y_val.shape)

Träning: (12725,)
Validering: (4960,)


## Baseline

Jag skapar en enkel baseline för att kunna jämföra med min ML-modell.

In [19]:

baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_val)

print("Baseline accuracy:", accuracy_score(y_val, baseline_pred))

Baseline accuracy: 0.5


## Logistisk regression

Jag tränar en logistisk regressionsmodell med fem finansiella nyckeltal.
Jag standardiserar datan så att nyckeltalen får jämförbara skalor.

In [20]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

## Utvärdera modellen

Jag testar modellen på 2017 och jämför resultatet med min baseline.

In [21]:
y_pred = model.predict(X_val_scaled)

accuracy = accuracy_score(y_val, y_pred)

print("Baseline accuracy:", 0.5)
print("Logistisk regression accuracy:", round(accuracy, 4))

Baseline accuracy: 0.5
Logistisk regression accuracy: 0.5228


## ROC-AUC

Jag undersöker hur bra modellen skiljer mellan de två klasserna.

In [22]:
y_prob = model.predict_proba(X_val_scaled)[:, 1]

auc = roc_auc_score(y_val, y_prob)

print("ROC-AUC:", round(auc, 4))

ROC-AUC: 0.5633


## Confusion matrix

Jag undersöker hur många förutsägelser modellen gör rätt och fel för varje klass.

In [23]:
cm = confusion_matrix(y_val, y_pred)

print(cm)

[[1471 1009]
 [1358 1122]]


## Random Forest

Jag testar Random Forest för att se om den ger bättre resultat än logistisk regression.

In [24]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

rf_model.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap:

## Utvärdera Random Forest

Jag testar Random Forest på 2017 och jämför resultatet med logistisk regression.

In [25]:
rf_pred = rf_model.predict(X_val)
rf_prob = rf_model.predict_proba(X_val)[:, 1]

rf_accuracy = accuracy_score(y_val, rf_pred)
rf_auc = roc_auc_score(y_val, rf_prob)

print("Random Forest accuracy:", round(rf_accuracy, 4))
print("Random Forest ROC-AUC:", round(rf_auc, 4))

Random Forest accuracy: 0.5149
Random Forest ROC-AUC: 0.5528


## Resultat och slutsats

Jag testade två ML-modeller med fem finansiella nyckeltal.

- Baseline accuracy: 50 %
- Logistisk regression accuracy: 52,28 %
- Logistisk regression ROC-AUC: 0,5633
- Random Forest accuracy: 51,49 %
- Random Forest ROC-AUC: 0,5528

Logistisk regression gav bäst resultat i detta experiment.

Båda modellerna presterade något bättre än baseline, men resultaten är fortfarande ganska svaga.

Jag har använt 2014–2016 för träning och 2017 för validering. År 2018 sparas till det slutliga testet.

## Undersökning av extremvärden med IQR

Jag undersöker om mina fem finansiella nyckeltal innehåller extremvärden som kan påverka modellernas resultat.

Jag använder IQR (Interquartile Range) för att identifiera värden som ligger ovanligt långt från den centrala delen av fördelningen.

Gränserna beräknas med Q1 − 1,5 × IQR och Q3 + 1,5 × IQR.

Undersökningen görs på träningsdata från 2014–2016 för att undvika dataläckage. I detta steg identifierar jag endast extremvärden och ändrar inga värden.

In [26]:

# Undersök extremvärden i träningsdatan (2014–2016)

for column in features:
    values = train_df[column].dropna()

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = ((values < lower) | (values > upper)).sum()

    print(f"{column}: {outliers} extremvärden av {len(values)}")


ROE: 2156 extremvärden av 11482
PE ratio: 746 extremvärden av 11562
Debt to Equity: 1351 extremvärden av 11497
Net Profit Margin: 2223 extremvärden av 11840
Gross Margin: 71 extremvärden av 12106


### Fördelning av extremvärden

IQR-analysen visar att framför allt ROE och Net Profit Margin har många observationer utanför de beräknade gränserna.

Jag undersöker därför hur stora de lägsta och högsta värdena är innan jag testar winsorisering. Syftet är att förstå hur extremvärdena kan påverka modellernas resultat.

In [27]:

# Visa fördelningen för de fem nyckeltalen

train_df[features].describe(
    percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
).T.round(2)


,count,mean,std,min,1%,25%,50%,75%,99%,max
ROE,11482.0,1578.93,104566.25,-969.39,-4.15,-0.06,0.07,0.14,4.69,8255949.33
PE ratio,11562.0,43.73,1108.81,0.00,0.00,0.00,13.78,23.51,285.15,105828.71
Debt to Equity,11497.0,0.46,33.61,-2586.44,-12.86,0.00,0.42,1.06,12.66,2131.51
Net Profit Margin,11840.0,-7.50,164.25,-9704.67,-74.75,-0.02,0.04,0.13,0.86,97.40
Gross Margin,12106.0,0.48,1.10,-74.32,-0.17,0.25,0.46,0.80,1.00,31.00


### Winsorisering med IQR

Undersökningen visar stora skillnader mellan medianvärdena och de högsta värdena, särskilt för ROE och P/E-tal.

Jag testar därför winsorisering med IQR för att begränsa extremvärden utan att ta bort observationer.

Gränserna beräknas enbart från träningsdata (2014–2016). Samma gränser ska sedan användas på valideringsdata (2017) för att undvika dataläckage.

Syftet är att undersöka om behandlingen av extremvärden förbättrar modellernas resultat.

In [28]:

# Skapa en kopia av träningsdatan
train_iqr = train_df.copy()

# Spara gränserna för varje nyckeltal
iqr_bounds = {}

for column in features:
    q1 = train_df[column].quantile(0.25)
    q3 = train_df[column].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    iqr_bounds[column] = (lower, upper)

    train_iqr[column] = train_iqr[column].clip(
        lower=lower,
        upper=upper
    )

print("Winsorisering klar!")
print("Original ROE max:", train_df["ROE"].max())
print("Winsoriserad ROE max:", train_iqr["ROE"].max())


Winsorisering klar!
Original ROE max: 8255949.3333
Winsoriserad ROE max: 0.4496374999999999


In [29]:

# Kopiera valideringsdatan
X_val_iqr = X_val.copy()

# Begränsa extremvärden med träningsdatans IQR-gränser
for i, column in enumerate(features):
    lower, upper = iqr_bounds[column]

    X_val_iqr[:, i] = np.clip(
        X_val_iqr[:, i],
        lower,
        upper
    )

print("Valideringsdata förberedd!")
print("Form:", X_val_iqr.shape)


Valideringsdata förberedd!
Form: (4960, 5)


In [30]:

# Skapa en kopia av träningsdatan
X_train_iqr = X_train.copy()

# Använd IQR-gränserna från träningsdatan
for i, column in enumerate(features):
    lower, upper = iqr_bounds[column]

    X_train_iqr[:, i] = np.clip(
        X_train_iqr[:, i],
        lower,
        upper
    )

print("Träningsdata förberedd!")
print("Form:", X_train_iqr.shape)
print("Saknade värden:", np.isnan(X_train_iqr).sum())


Träningsdata förberedd!
Form: (12725, 5)
Saknade värden: 0


### Utvärdering av logistisk regression efter IQR-winsorisering

Jag tränar om den logistiska regressionsmodellen med de winsoriserade nyckeltalen.

Syftet är att undersöka om begränsningen av extremvärden påverkar modellens förmåga att klassificera aktier utifrån nästa års avkastning.

Jag använder samma träningsperiod (2014–2016), valideringsperiod (2017) och målvariabel som tidigare. Resultaten jämförs med ursprungsmodellens accuracy och ROC-AUC.

In [31]:
# Skala den winsoriserade datan
scaler_iqr = StandardScaler()

X_train_iqr_scaled = scaler_iqr.fit_transform(X_train_iqr)
X_val_iqr_scaled = scaler_iqr.transform(X_val_iqr)

# Träna logistisk regression
logreg_iqr = LogisticRegression(max_iter=1000)
logreg_iqr.fit(X_train_iqr_scaled, y_train)

# Gör prediktioner på 2017
y_pred_iqr = logreg_iqr.predict(X_val_iqr_scaled)
y_prob_iqr = logreg_iqr.predict_proba(X_val_iqr_scaled)[:, 1]

# Utvärdera modellen
accuracy_iqr = accuracy_score(y_val, y_pred_iqr)
auc_iqr = roc_auc_score(y_val, y_prob_iqr)

print(f"Accuracy efter IQR: {accuracy_iqr:.2%}")
print(f"ROC-AUC efter IQR: {auc_iqr:.4f}")


Accuracy efter IQR: 59.54%
ROC-AUC efter IQR: 0.5926


In [32]:

# Kontrollera hur många värden som ändrats av IQR

for i, column in enumerate(features):
    changed = np.sum(X_train[:, i] != X_train_iqr[:, i])
    print(f"{column}: {changed} ändrade värden")


ROE: 2156 ändrade värden
PE ratio: 746 ändrade värden
Debt to Equity: 1351 ändrade värden
Net Profit Margin: 2223 ändrade värden
Gross Margin: 71 ändrade värden


### Winsorisering med 1:a och 99:e percentilen

Jag jämför IQR-baserad winsorisering med en metod som begränsar värden under den 1:a percentilen och över den 99:e percentilen.

Gränserna beräknas från träningsdata (2014–2016) och används sedan på valideringsdata (2017).

Syftet är att undersöka om en mindre omfattande begränsning av extremvärden ger bättre resultat än IQR-metoden.

In [33]:

# Skapa kopior av ursprungliga modellvariabler
X_train_pct = X_train.copy()
X_val_pct = X_val.copy()

# Beräkna percentilgränser på träningsdatan
pct_bounds = {}

for i, column in enumerate(features):
    lower = train_df[column].quantile(0.01)
    upper = train_df[column].quantile(0.99)

    pct_bounds[column] = (lower, upper)

    X_train_pct[:, i] = np.clip(
        X_train_pct[:, i], lower, upper
    )

    X_val_pct[:, i] = np.clip(
        X_val_pct[:, i], lower, upper
    )

    changed = np.sum(X_train[:, i] != X_train_pct[:, i])
    print(f"{column}: {changed} ändrade värden")

print("Percentilbaserad winsorisering klar!")


ROE: 230 ändrade värden
PE ratio: 116 ändrade värden
Debt to Equity: 230 ändrade värden
Net Profit Margin: 238 ändrade värden
Gross Margin: 159 ändrade värden
Percentilbaserad winsorisering klar!


### Utvärdering av logistisk regression efter percentilbaserad winsorisering

Jag tränar om den logistiska regressionsmodellen med finansiella nyckeltal som begränsats vid den 1:a och 99:e percentilen.

Jag använder samma tränings- och valideringsperioder som tidigare och jämför resultaten med både ursprungsmodellen och IQR-winsoriseringen.

Syftet är att undersöka vilken metod som ger bäst resultat på valideringsdata från 2017.

In [34]:

# Skala datan efter percentilbaserad winsorisering
scaler_pct = StandardScaler()

X_train_pct_scaled = scaler_pct.fit_transform(X_train_pct)
X_val_pct_scaled = scaler_pct.transform(X_val_pct)

# Träna logistisk regression
logreg_pct = LogisticRegression(max_iter=1000)
logreg_pct.fit(X_train_pct_scaled, y_train)

# Prediktioner på valideringsdata (2017)
y_pred_pct = logreg_pct.predict(X_val_pct_scaled)
y_prob_pct = logreg_pct.predict_proba(X_val_pct_scaled)[:, 1]

# Beräkna resultat
accuracy_pct = accuracy_score(y_val, y_pred_pct)
auc_pct = roc_auc_score(y_val, y_prob_pct)

print(f"Accuracy efter percentiler: {accuracy_pct:.2%}")
print(f"ROC-AUC efter percentiler: {auc_pct:.4f}")


Accuracy efter percentiler: 54.33%
ROC-AUC efter percentiler: 0.5892


### Utvärdering av Random Forest efter IQR-winsorisering

Jag undersöker om IQR-winsorisering även påverkar resultatet för Random Forest.

Random Forest använder flera beslutsträd och är generellt mindre känslig för extremvärden än logistisk regression.

Jag använder samma modellinställningar, träningsdata (2014–2016) och valideringsdata (2017) som tidigare.

Syftet är att jämföra modellens accuracy och ROC-AUC före och efter winsorisering.

In [35]:
# Träna Random Forest med IQR-winsoriserad data
rf_iqr = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

rf_iqr.fit(X_train_iqr, y_train)

# Prediktioner på valideringsdata (2017)
rf_pred_iqr = rf_iqr.predict(X_val_iqr)
rf_prob_iqr = rf_iqr.predict_proba(X_val_iqr)[:, 1]

# Utvärdera modellen
rf_accuracy_iqr = accuracy_score(y_val, rf_pred_iqr)
rf_auc_iqr = roc_auc_score(y_val, rf_prob_iqr)

print(f"Random Forest accuracy efter IQR: {rf_accuracy_iqr:.2%}")
print(f"Random Forest ROC-AUC efter IQR: {rf_auc_iqr:.4f}")


Random Forest accuracy efter IQR: 51.43%
Random Forest ROC-AUC efter IQR: 0.5516


### Sammanfattning av modellresultat

Tabellen nedan sammanfattar resultaten från de olika modellerna och metoderna för att hantera extremvärden. Samtliga modeller utvärderas på valideringsdata från 2017.

In [36]:

# Sammanställ resultaten från experimenten

results = pd.DataFrame({
    "Modell": [
        "Logistisk regression",
        "Logistisk regression",
        "Logistisk regression",
        "Random Forest",
        "Random Forest"
    ],
    "Metod": [
        "Utan winsorisering",
        "IQR",
        "Percentiler 1–99 %",
        "Utan winsorisering",
        "IQR"
    ],
    "Accuracy (%)": [
        accuracy * 100,
        accuracy_iqr * 100,
        accuracy_pct * 100,
        rf_accuracy * 100,
        rf_accuracy_iqr * 100
    ],
    "ROC-AUC": [
        auc,
        auc_iqr,
        auc_pct,
        rf_auc,
        rf_auc_iqr
    ]
})

results.round(4)


,Modell,Metod,Accuracy (%),ROC-AUC
0,Logistisk regression,Utan winsorisering,52.2782,0.5633
1,Logistisk regression,IQR,59.5363,0.5926
2,Logistisk regression,Percentiler 1–99 %,54.3347,0.5892
3,Random Forest,Utan winsorisering,51.4919,0.5528
4,Random Forest,IQR,51.4315,0.5516


### Resultat och slutsats – hantering av extremvärden

Jag undersökte hur extremvärden i fem finansiella nyckeltal påverkade logistisk regression och Random Forest.

IQR-analysen visade att framför allt ROE och Net Profit Margin innehöll många värden utanför de beräknade gränserna.

För logistisk regression ökade accuracy från 52,28 % till 59,54 % efter IQR-winsorisering. ROC-AUC ökade från 0,5633 till 0,5926.

Winsorisering vid den 1:a och 99:e percentilen gav också en förbättring, men inte lika stor: 54,33 % accuracy och 0,5892 ROC-AUC.

För Random Forest förändrades resultatet däremot knappt. Accuracy gick från 51,49 % till 51,43 % och ROC-AUC från 0,5528 till 0,5516.

Resultaten tyder på att logistisk regression är mer känslig för extremvärden i denna datamängd än Random Forest. IQR gav bäst resultat av de testade alternativen för logistisk regression.

IQR-winsorisering gav högst accuracy för logistisk regression, men metoden begränsade också en relativt stor andel av observationerna, särskilt för ROE och Net Profit Margin.

Resultatet visar att hantering av extremvärden kan påverka modellens prestanda, men en högre accuracy innebär inte automatiskt att modellen är bättre på att identifiera lönsamma aktier.

Därför behöver resultatet tolkas försiktigt och utvärderas vidare på tidigare oanvänd testdata.

## Rangordning av aktier – Top 10 % och Bottom 10 %

Jag undersöker om modellen kan rangordna aktier utifrån deras förväntade prestation.

Jag använder sannolikheterna från logistisk regression med IQR-winsorisering för valideringsåret 2017.

Syftet är att jämföra den faktiska genomsnittliga avkastningen för de 10 % högst rankade aktierna med de 10 % lägst rankade aktierna.

Detta är ett komplement till accuracy och ROC-AUC och visar hur modellens rangordning förhåller sig till faktisk avkastning.

In [37]:

# Kontrollera avkastningskolumnen för 2017
print([
    col for col in df_2017.columns
    if "PRICE VAR" in col.upper()
    or "RETURN" in col.upper()
])

print("Antal rader 2017:", len(df_2017))
print("Antal prediktioner:", len(y_prob_iqr))


['returnOnAssets', 'returnOnEquity', 'returnOnCapitalEmployed', 'Return on Tangible Assets', 'Next_Year_Return']
Antal rader 2017: 4960
Antal prediktioner: 4960


In [38]:

# Visa kolumner som innehåller PRICE VAR
for col in df_2017.columns:
    if "PRICE VAR" in col.upper():
        print(repr(col))

# Kontrollera vilka kolumner som används i valideringen
print("Valideringsdata:", X_val.shape)
print("Avkastningsdata:", df_2017.shape)


Valideringsdata: (4960, 5)
Avkastningsdata: (4960, 226)


In [39]:

# Visa kolumnerna längst bak i 2017 års data
print(df_2017.columns[-15:].tolist())

# Kontrollera om Target och avkastning finns
print("Kolumner i train_df:")
print([
    col for col in train_df.columns
    if "target" in col.lower()
    or "return" in col.lower()
    or "price" in col.lower()
])


['3Y Shareholders Equity Growth (per Share)', '10Y Dividend per Share Growth (per Share)', '5Y Dividend per Share Growth (per Share)', '3Y Dividend per Share Growth (per Share)', 'Receivables growth', 'Inventory Growth', 'Asset Growth', 'Book Value per Share Growth', 'Debt Growth', 'R&D Expense Growth', 'SG&A Expenses Growth', 'Sector', 'Next_Year_Return', 'Class', 'Target']
Kolumner i train_df:
['priceBookValueRatio', 'priceToBookRatio', 'priceToSalesRatio', 'priceEarningsRatio', 'priceToFreeCashFlowsRatio', 'priceToOperatingCashFlowsRatio', 'priceCashFlowRatio', 'priceEarningsToGrowthRatio', 'priceSalesRatio', 'priceFairValue', 'returnOnAssets', 'returnOnEquity', 'returnOnCapitalEmployed', 'Price to Sales Ratio', 'Return on Tangible Assets', 'Next_Year_Return', 'Target']


In [40]:

# Kontrollera var nästa års avkastning finns
for name, df in [
    ("df_2017", df_2017),
    ("train_df", train_df)
]:
    print(f"\n{name}:")
    for col in df.columns:
        if "Next_Year" in col or "PRICE VAR" in col.upper():
            print(repr(col))



df_2017:
'Next_Year_Return'

train_df:
'Next_Year_Return'


In [41]:

# Kontrollera att valideringsdatan matchar 2017 års aktier

print("Antal rader:", len(df_2017))
print("Antal prediktioner:", len(y_prob_iqr))

# Visa första fem värdena för varje nyckeltal
for i, column in enumerate(features):
    original = df_2017[column].head(5).to_numpy()
    model = X_val[:5, i]

    print(f"\n{column}")
    print("Original:", original)
    print("Modell:  ", model)


Antal rader: 4960
Antal prediktioner: 4960

ROE
Original: [0.2748 0.1364 0.2949 0.0066 0.383 ]
Modell:   [0.2748 0.1364 0.2949 0.0066 0.383 ]

PE ratio
Original: [  15.0259   24.9252   16.1538 1307.5      20.0603]
Modell:   [  15.0259   24.9252   16.1538 1307.5      20.0603]

Debt to Equity
Original: [0.5664 0.4031 2.1017 5.359  2.1908]
Modell:   [0.5664 0.4031 2.1017 5.359  2.1908]

Net Profit Margin
Original: [2.356e-01 2.670e-02 1.710e-02 2.000e-04 1.061e-01]
Modell:   [2.356e-01 2.670e-02 1.710e-02 2.000e-04 1.061e-01]

Gross Margin
Original: [0.4983 0.2235 0.224  0.2209 0.3565]
Modell:   [0.4983 0.2235 0.224  0.2209 0.3565]


In [42]:

# Kontrollera alla rader och alla fem nyckeltal
# NaN i originaldatan ignoreras eftersom modellen
# redan har ersatt saknade värden.

original = df_2017[features].to_numpy()

matchar = np.isclose(
    original,
    X_val,
    equal_nan=True
)

# Kontrollera bara värden som inte saknas i originalet
finns_varde = ~np.isnan(original)

antal_fel = np.sum(~matchar & finns_varde)

print("Antal avvikande värden:", antal_fel)
print("Antal prediktioner:", len(y_prob_iqr))
print("Antal avkastningsvärden:", len(df_2017["Next_Year_Return"]))
print(
    "Saknade avkastningsvärden:",
    df_2017["Next_Year_Return"].isna().sum()
)


Antal avvikande värden: 0
Antal prediktioner: 4960
Antal avkastningsvärden: 4960
Saknade avkastningsvärden: 0


### Jämförelse av de 10 % högst och lägst rankade aktierna

Jag använder sannolikheterna från logistisk regression med IQR-winsorisering för att rangordna aktierna i valideringsåret 2017.

Jag jämför sedan den faktiska genomsnittliga avkastningen för de 10 % högst rankade aktierna med de 10 % lägst rankade aktierna.

Syftet är att undersöka om modellens rangordning har ett samband med aktiernas faktiska avkastning.

In [43]:

# Rangordna aktier efter modellens sannolikheter

ranking = pd.DataFrame({
    "Sannolikhet": np.asarray(y_prob_iqr),
    "Faktisk_avkastning": df_2017["Next_Year_Return"].to_numpy()
})

# Sortera från högst till lägst sannolikhet
ranking = ranking.sort_values(
    "Sannolikhet", ascending=False
)

# Välj 10 % av aktierna i varje grupp
n = int(len(ranking) * 0.10)

top_10 = ranking.head(n)
bottom_10 = ranking.tail(n)

# Beräkna genomsnittlig faktisk avkastning
top_return = top_10["Faktisk_avkastning"].mean()
bottom_return = bottom_10["Faktisk_avkastning"].mean()

print("Antal aktier per grupp:", n)
print(f"Top 10 % genomsnittlig avkastning: {top_return:.2f} %")
print(f"Bottom 10 % genomsnittlig avkastning: {bottom_return:.2f} %")
print(f"Skillnad: {top_return - bottom_return:.2f} procentenheter")


Antal aktier per grupp: 496
Top 10 % genomsnittlig avkastning: -10.42 %
Bottom 10 % genomsnittlig avkastning: -23.61 %
Skillnad: 13.19 procentenheter


### Resultat – Top 10 % och Bottom 10 %

Jag jämförde den faktiska genomsnittliga avkastningen för de 10 % högst respektive lägst rankade aktierna under valideringsåret 2017.

De 496 högst rankade aktierna hade en genomsnittlig avkastning på -10,42 %, medan de 496 lägst rankade aktierna hade -23,61 %.

Skillnaden var 13,19 procentenheter till fördel för de högst rankade aktierna.

Resultatet tyder på att modellens rangordning hade ett visst samband med den faktiska avkastningen under valideringsåret. Båda grupperna hade dock negativ genomsnittlig avkastning.

Resultatet gäller endast 2017 och visar inte att modellen är en lönsam investeringsstrategi. Ytterligare utvärdering behövs för att bedöma hur väl modellen generaliserar.

### Kontroll av extremvärden i aktieavkastningen

Jag undersöker om enskilda aktier med mycket stora positiva eller negativa avkastningar påverkar jämförelsen mellan de högst och lägst rankade aktierna.

Jag jämför medelvärde och median samt kontrollerar lägsta och högsta avkastning.

Detta är en kompletterande analys. Jag ändrar inte de ursprungliga avkastningsvärdena.

In [44]:

# Undersök avkastningens fördelning i båda grupperna

for namn, grupp in [
    ("Top 10 %", top_10),
    ("Bottom 10 %", bottom_10)
]:
    avkastning = grupp["Faktisk_avkastning"]

    print(f"\n{namn}")
    print(f"Medelvärde: {avkastning.mean():.2f} %")
    print(f"Median: {avkastning.median():.2f} %")
    print(f"Min: {avkastning.min():.2f} %")
    print(f"Max: {avkastning.max():.2f} %")
    print(f"Under -100 %: {(avkastning < -100).sum()} aktier")



Top 10 %
Medelvärde: -10.42 %
Median: -9.62 %
Min: -97.85 %
Max: 150.08 %
Under -100 %: 0 aktier

Bottom 10 %
Medelvärde: -23.61 %
Median: -35.36 %
Min: -100.00 %
Max: 378.18 %
Under -100 %: 0 aktier


### Kompletterande analys av avkastningen

Jag jämförde även medianavkastningen för att undersöka om extrema avkastningar påverkade resultatet.

Medianavkastningen var -9,62 % för Top 10 % och -35,36 % för Bottom 10 %, en skillnad på 25,74 procentenheter.

Ingen av grupperna innehöll avkastningar under -100 %. Däremot fanns stora positiva avkastningar, särskilt i Bottom 10 %, vilket kan påverka medelvärdet.

Både medelvärdet och medianen visar att de högst rankade aktierna hade bättre avkastning än de lägst rankade i denna validering. Resultatet behöver dock undersökas på ytterligare data innan några generella slutsatser kan dras.

## Experiment med XGBoost

Jag testar XGBoost som en alternativ maskininlärningsmodell för att undersöka om en gradientboostad trädmodell kan förbättra klassificeringen av aktier.

Jag använder samma fem finansiella nyckeltal som tidigare och behåller den tidsbaserade uppdelningen:

- Träningsdata: 2014–2016
- Valideringsdata: 2017
- Slutlig testdata: 2018 (används inte i detta experiment)

Jag kommer att jämföra XGBoost med logistisk regression och Random Forest med hjälp av accuracy och ROC-AUC.

In [45]:

# XGBoost – första experimentet
# Träning: 2014–2016
# Validering: 2017

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

# Träna modellen
xgb_model.fit(X_train, y_train)

print("XGBoost är färdigtränad!")


XGBoost är färdigtränad!


In [46]:
# Gör förutsägelser på 2017 års data
y_pred_xgb = xgb_model.predict(X_val)

# Sannolikheten för klass 1
y_prob_xgb = xgb_model.predict_proba(X_val)[:, 1]

# Beräkna resultat
accuracy_xgb = accuracy_score(y_val, y_pred_xgb)
auc_xgb = roc_auc_score(y_val, y_prob_xgb)

print(f"XGBoost Accuracy: {accuracy_xgb:.2%}")
print(f"XGBoost ROC-AUC: {auc_xgb:.4f}")


XGBoost Accuracy: 51.57%
XGBoost ROC-AUC: 0.5434


In [47]:
# Anpassa skalningen endast på träningsdata
robust_scaler = RobustScaler()

X_train_robust = robust_scaler.fit_transform(X_train)
X_val_robust = robust_scaler.transform(X_val)

print("RobustScaler är klar!")
print("Träningsdata:", X_train_robust.shape)
print("Valideringsdata:", X_val_robust.shape)


RobustScaler är klar!
Träningsdata: (12725, 5)
Valideringsdata: (4960, 5)


In [48]:
# Träna logistisk regression med RobustScaler
logreg_robust = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logreg_robust.fit(X_train_robust, y_train)

# Gör förutsägelser på 2017
y_pred_robust = logreg_robust.predict(X_val_robust)
y_prob_robust = logreg_robust.predict_proba(X_val_robust)[:, 1]

# Utvärdera modellen
accuracy_robust = accuracy_score(y_val, y_pred_robust)
auc_robust = roc_auc_score(y_val, y_prob_robust)

print(f"RobustScaler Accuracy: {accuracy_robust:.2%}")
print(f"RobustScaler ROC-AUC: {auc_robust:.4f}")


RobustScaler Accuracy: 52.30%
RobustScaler ROC-AUC: 0.5633


### Jämförelse med XGBoost och RobustScaler

Jag testade två ytterligare metoder för att undersöka om klassificeringen kunde förbättras.

**XGBoost:** Modellen tränades på samma fem finansiella nyckeltal som tidigare och utvärderades på 2017 års data. Resultatet blev 51,57 % accuracy och 0,5434 ROC-AUC.

**RobustScaler:** Jag testade också logistisk regression med RobustScaler, som använder median och kvartilavstånd vid skalning. Resultatet blev 52,30 % accuracy och 0,5633 ROC-AUC.

**Slutsats:** Ingen av dessa metoder förbättrade resultatet jämfört med logistisk regression med IQR-winsorisering, som uppnådde 59,54 % accuracy och 0,5926 ROC-AUC.

Resultaten gäller valideringsåret 2017. Innan modellen väljs för slutlig testning på 2018 behöver experimenten kontrolleras och jämföras inom gruppen.

In [49]:

# Kontrollera klassfördelningen och IQR-modellens prediktioner

print("Faktisk klassfördelning 2017:")
print("Klass 0:", np.sum(y_val == 0))
print("Klass 1:", np.sum(y_val == 1))

print("\nIQR-modellens prediktioner:")
print("Klass 0:", np.sum(y_pred_iqr == 0))
print("Klass 1:", np.sum(y_pred_iqr == 1))

print("\nConfusion matrix:")
print(confusion_matrix(y_val, y_pred_iqr))


Faktisk klassfördelning 2017:
Klass 0: 2480
Klass 1: 2480

IQR-modellens prediktioner:
Klass 0: 2075
Klass 1: 2885

Confusion matrix:
[[1274 1206]
 [ 801 1679]]


In [50]:

print("Ursprunglig logistisk regression:")
print(classification_report(y_val, y_pred))

print("Logistisk regression med IQR:")
print(classification_report(y_val, y_pred_iqr))


Ursprunglig logistisk regression:
              precision    recall  f1-score   support

           0       0.52      0.59      0.55      2480
           1       0.53      0.45      0.49      2480

    accuracy                           0.52      4960
   macro avg       0.52      0.52      0.52      4960
weighted avg       0.52      0.52      0.52      4960

Logistisk regression med IQR:
              precision    recall  f1-score   support

           0       0.61      0.51      0.56      2480
           1       0.58      0.68      0.63      2480

    accuracy                           0.60      4960
   macro avg       0.60      0.60      0.59      4960
weighted avg       0.60      0.60      0.59      4960




### Kontroll av IQR-modellens klassificering

Jag undersökte klassfördelningen och jämförde precision, recall och F1-score för den ursprungliga logistiska regressionen och modellen med IQR-winsorisering.

Valideringsdata från 2017 innehåller 2 480 observationer i vardera klass.

Efter IQR-winsorisering ökade accuracy från 52,28 % till 59,54 % och ROC-AUC från 0,5633 till 0,5926.

För klass 1 ökade recall från 0,45 till 0,68 och F1-score från 0,49 till 0,63.

Resultaten visar att förbättringen inte enbart gäller accuracy. Modellen identifierar också en större andel av de aktier som överträffade årets medianavkastning.

Resultaten är baserade på valideringsåret 2017 och behöver kontrolleras på oberoende testdata innan några generella slutsatser dras.
